In [1]:
# Install LangChain and Ollama integration
!pip install -q langchain langchain-ollama


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import ChatOllama to communicate with a locally running LLM
from langchain_ollama import ChatOllama

### Initialize the LLM

In [3]:
# Create an LLM object using the Ollama model

llm = ChatOllama(
    model="llama3",      # Local model downloaded using Ollama
    temperature=0         # Lower temperature = more deterministic responses
)

### **LLM CALL**

In [4]:
# Invoke (send) a prompt to the LLM
response = llm.invoke(
    "What is LLM? Answer in 10 words."
)

# Print only the text/content returned by the model
print(response.content)

Large Language Model: AI-powered text generator for natural language processing.


In [5]:
# Import the requests library to make HTTP requests
import requests

# Send a GET request to the Exchange Rate API
# This returns the latest exchange rates with USD as the base currency
rate = requests.get("https://open.er-api.com/v6/latest/USD").json()

# Extract the INR exchange rate from the JSON response
# 'rates' is a dictionary containing all supported currencies
current_rate = rate["rates"]["INR"]

# Print the USD to INR exchange rate
print(current_rate)

95.766882


In [6]:
# Send a prompt to the LLM using the latest exchange rate
response = llm.invoke(
    f"""
    Current USD to INR exchange rate is {current_rate}.

    Explain this exchange rate in one simple sentence.
    """
)

# Print only the text generated by the LLM
print(response.content)

As of now, 1 US Dollar (USD) can be exchanged for approximately 95 Indian Rupees (INR).


In [7]:
# Install LangChain, Ollama integration, and DuckDuckGo search
!pip install -q langchain langchain-ollama langchain-community duckduckgo-search


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Import the DuckDuckGo search tool
from langchain_community.tools import DuckDuckGoSearchResults

# Create an instance of the DuckDuckGo search tool
search = DuckDuckGoSearchResults()
# Search for the latest AI news
news = search.invoke("top AI news today")
# Ask the LLM to summarize the search results
response = llm.invoke(
    f"""
    Summarize the following AI news in simple language:

    {news}
    """
)
# Print the generated summary
print(response.content)

C:\Users\anubh\AppData\Local\Temp\ipykernel_40128\389323670.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


Here are the summaries of the AI news snippets in simple language:

1. **Google News - Artificial Intelligence**: This is a news aggregator that collects articles and videos about artificial intelligence (AI) from various sources. You can read the latest news, browse thousands of titles, and watch videos on AI-related topics.
2. **AI News & Artificial Intelligence | TechCrunch**: This is a section on the TechCrunch website dedicated to covering the latest news and trends in artificial intelligence and machine learning technology. It also explores the companies building these technologies and the ethical issues surrounding AI.
3. **AI News Today, August 19 — Top AI Stories & Live Updates**: This is a daily update of top AI stories from around the world. The article provides summaries of recent developments in AI, including model releases, pricing changes, funding news, and breaking stories from companies like OpenAI, Anthropic, Google, xAI, and Meta.
4. **Daily AI News — Today's AI Upda

In [9]:
pip install -U ddgs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Import the requests library to make HTTP requests
import requests

# Open-Meteo provides free weather APIs without requiring an API key.
# Latitude and longitude below correspond to New Delhi.
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=19.9975"
    "&longitude=73.7898"
    "&current=temperature_2m"
)

# Send a GET request to the weather API.
# Convert the JSON response into a Python dictionary.
weather_data = requests.get(url).json()

# Extract the current temperature (2 meters above ground level)
temperature = weather_data["current"]["temperature_2m"]

# Send the temperature to the LLM for explanation
response = llm.invoke(
    f"""
    Current temperature is {temperature}°C.

    Describe the weather in one simple sentence.
    """
)

# Print only the generated response
print(response.content)

The weather is warm and pleasant, with a current temperature of 23.4°C.


In [11]:
# Used to call REST APIs
import requests

# Used to convert a normal Python function into a LangChain Tool
from langchain_core.tools import tool

In [12]:
@tool
def get_temperature():
    """
    Get the current temperature in Nashik.
    """

    # Weather API URL
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=19.9975"
        "&longitude=73.7898"
        "&current=temperature_2m"
    )

    # Fetch the weather data
    weather_data = requests.get(url).json()

    # Return only the current temperature as a string
    return str(weather_data["current"]["temperature_2m"])

@tool
def get_usd_inr() -> str:
    """
    Returns the latest USD to INR exchange rate.
    Use this whenever the user asks about currency conversion.
    """

    url = "https://open.er-api.com/v6/latest/USD"

    response = requests.get(url)

    exchange_data = response.json()

    exchange_rate = exchange_data["rates"]["INR"]

    return str(exchange_rate)


@tool
def get_ai_news() -> str:
    """
    Returns latest AI news from DuckDuckGo search.
    """

    return search.invoke("latest AI news")

In [13]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",   # Tool-calling capable model
    temperature=0
)

In [14]:
# Bind (attach) the tools to the LLM
tool_llm = llm.bind_tools(
    [
        get_temperature,
        get_usd_inr,
        get_ai_news
    ]
)



In [15]:
# User's question
question = "What is the temperature outside?"

# Send the question to the LLM that has tools attached
response = tool_llm.invoke(question)

# Print the tool calls suggested by the LLM
print(response.tool_calls)

[{'name': 'get_temperature', 'args': {'city': 'Nashik'}, 'id': '05089bda-e224-401c-bf81-74fc4a331e35', 'type': 'tool_call'}]


In [16]:
from langchain.agents import create_agent

# Create an agent using our Ollama LLM and existing tools
agent = create_agent(
    model=llm,
    tools=[
        get_temperature,
        get_usd_inr,
        get_ai_news
    ]
)

In [17]:
# User's question
question = "What is today's USD to INR exchange rate?"

# Run the agent
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content="What is today's USD to INR exchange rate?", additional_kwargs={}, response_metadata={}, id='bb1e901e-07cb-4094-8203-734089910ff6'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-08-19T14:49:54.2302046Z', 'done': True, 'done_reason': 'stop', 'total_duration': 763779000, 'load_duration': 142075600, 'prompt_eval_count': 232, 'prompt_eval_duration': 74825400, 'eval_count': 19, 'eval_duration': 492058300, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a01a80-0839-74f1-ab07-ac0d992e74a4-0', tool_calls=[{'name': 'get_usd_inr', 'args': {}, 'id': '9b7fc573-6665-46ce-9b66-ae1fdf503759', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 232, 'output_tokens': 19, 'total_tokens': 251}), ToolMessage(content='95.766882', name='get_usd_inr', id='1469e28d-9ace-4139-abd3-cf2fce5bd23c', tool_call_id='9b7fc573-6665-46ce-9b66-ae1fdf503759'), A

In [18]:
print(response["messages"][-1].content)

Today's USD to INR exchange rate is approximately 95.77 (rounded to two decimal places).


In [19]:
#Memory concept
response = llm.invoke(
    "my special please."
)

print(response.content)

Of course! Could you please provide more details about what you'd like me to assist you with? That way, I can better understand and fulfill your request.


In [20]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

# Send system instruction + user message together
response = llm.invoke(
    [
        # Defines how the AI should behave
        SystemMessage(
            content="You are a polite chai seller."
        ),

        # User's actual message with context
        HumanMessage(
            content="I am your regular customer. I love your special ginger tea. My special please."
        )
    ]
)

# Print LLM response
print(response.content)

Of course, right away! For you, it's a special ginger tea with just the right amount of warmth and spice. Coming up just for you! Enjoy your favorite blend!


In [21]:
# First call: Tell the LLM your favorite tea
print(
    llm.invoke(
        "My favorite tea is ginger tea."
    ).content
)

# Second call: Ask the LLM about the information
# given in the previous call
print(
    llm.invoke(
        "What is my favorite tea?"
    ).content
)

That's great! Ginger tea can be a delightful and warming beverage, especially during colder months or when you're feeling under the weather. It has natural properties that can help soothe an upset stomach and provide relief from nausea. Would you like to know more about how to make ginger tea or its potential health benefits?
To determine your favorite tea, I would need more information about you and your preferences. However, since I don't have any personal data or previous interactions with you, I can't provide an accurate answer.

If you'd like to share some details about the types of teas you enjoy, such as black tea, green tea, herbal tea, or specific flavors (like mint, chamomile, or jasmine), that would help me suggest a favorite based on common preferences. Alternatively, if you've tried various types and have a particular one you return to often, I can offer some insights into why it might be your favorite!


In [22]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create in-memory short-term memory
memory = InMemorySaver()

# Create agent with Ollama model + memory
conversation = create_agent(
    model=llm,
    tools=[],
    checkpointer=memory
)

# Same thread_id = same conversation memory
config = {
    "configurable": {
        "thread_id": "tea_conversation"
    }
}

# First message
response1 = conversation.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My favorite tea is ginger tea."
            }
        ]
    },
    config=config
)

print(response1["messages"][-1].content)

# Second message
response2 = conversation.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my favorite tea?"
            }
        ]
    },
    config=config
)

print(response2["messages"][-1].content)

That's great to hear! Ginger tea can be a delightful and warming beverage, especially during colder months or when you're feeling under the weather. It has natural properties that can help soothe an upset stomach and provide some relief from nausea.

If you'd like to know more about how to make ginger tea or any variations of it, feel free to ask! Here's a simple recipe for you:

### Basic Ginger Tea Recipe

**Ingredients:**
- 1-inch piece of fresh ginger
- 2 cups water
- Honey or sugar (optional)
- Lemon slices (optional)

**Instructions:**
1. Peel the ginger and cut it into thin slices.
2. Place the ginger in a pot with 2 cups of water.
3. Bring to a boil over medium heat, then reduce the heat to low and let it simmer for about 5 minutes.
4. Remove from heat and strain out the ginger pieces.
5. Add honey or sugar if you like it sweet, and lemon slices for some extra flavor.
6. Serve hot.

Feel free to adjust the sweetness and add other ingredients according to your taste preferences!

In [23]:
# First independent LLM call
print(
    llm.invoke(
        "Explain chai simply for a beginner in 1 line"
    ).content
)


Chai is a spicy Indian tea made with black tea, milk, and a blend of spices like cinnamon, cardamom, and ginger.


In [24]:
# Second independent LLM call
print(
    llm.invoke(
        "Explain LangChain simply for a beginner in 1 line"
    ).content
)

LangChain is a framework that helps build applications using language models to chain together sequences of interactions and tasks.


In [25]:
from langchain_core.prompts import ChatPromptTemplate

# Create a reusable prompt template
# {topic} is a dynamic variable/place-holder
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} simply for a beginner in 1 line"
)

# Create a chain:
# Prompt output → passed to → LLM
chain = prompt | llm


# Run same chain with different inputs
print(
    chain.invoke({"topic": "chai"}).content
)

print(
    chain.invoke({"topic": "LangChain"}).content
)

Chai is a spicy Indian tea made with black tea, milk, and a blend of spices like cinnamon, cardamom, and ginger.
LangChain is a framework that helps build applications using language models to chain together sequences of interactions and tasks.


In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Create a prompt template with a dynamic variable {topic}
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in 1 line"
)

# Fill {topic} with "LangChain"
# Then convert the generated prompt into chat messages
messages = prompt.invoke(
    {"topic": "LangChain"}
).to_messages()

# See the actual messages generated by the prompt template
print(messages)

# Send those messages to the Ollama LLM
response = llm.invoke(messages)

# Print only the generated text
print(response.content)

[HumanMessage(content='Explain LangChain in 1 line', additional_kwargs={}, response_metadata={})]
LangChain is a framework for building and chaining together language models to solve complex natural language processing tasks.


In [27]:
print(
    llm.invoke(
        "Write a title and a 3-line blog about AI in one response."
    ).content
)

Title: "The Future is Now: How Artificial Intelligence Is Transforming Our World"

Blog snippet: Embracing the power of artificial intelligence, we're witnessing unprecedented advancements that are reshaping industries from healthcare to entertainment. With smarter algorithms and increased data processing capabilities, AI is not just a future prospect but an integral part of our present, driving innovation and efficiency like never before.


In [28]:
from langchain_core.prompts import ChatPromptTemplate


# -----------------------------
# STEP 1: Create Prompt Templates
# -----------------------------

# First prompt generates a title
title_prompt = ChatPromptTemplate.from_template(
    "Write a short title about {topic}."
)

# Second prompt generates a blog using the title
# produced by the first chain
blog_prompt = ChatPromptTemplate.from_template(
    "Write a 3-line blog using this title: {title}"
)


# -----------------------------
# STEP 2: Create Chains
# -----------------------------

# Topic → Title Prompt → Ollama
title_chain = title_prompt | llm

# Title → Blog Prompt → Ollama
blog_chain = blog_prompt | llm


# -----------------------------
# STEP 3: Execute First Chain
# -----------------------------

title = title_chain.invoke(
    {"topic": "AI"}
).content


# -----------------------------
# STEP 4: Pass Output to Next Chain
# -----------------------------

blog = blog_chain.invoke(
    {"title": title}
).content


# -----------------------------
# STEP 5: Print Results
# -----------------------------

print(title)
print(blog)

"AI Revolution: Shaping the Future of Technology"
AI Revolution: Shaping the Future of Technology  
In the era of rapid technological advancement, artificial intelligence (AI) stands at the forefront, transforming industries and paving the way for unprecedented innovations. From healthcare to autonomous vehicles, AI's impact is reshaping our world in profound ways.

As we stand on the brink of this revolution, the potential for AI to solve complex global challenges and enhance human capabilities is both thrilling and daunting.


## CHAINS

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


# -----------------------------------
# 1. Prompt for generating the title
# -----------------------------------

title_prompt = ChatPromptTemplate.from_template(
    "Write a short title about {topic}"
)

# topic -> prompt -> Ollama -> title text
title_chain = title_prompt | llm


# -----------------------------------
# 2. Prompt for generating the blog
# -----------------------------------

blog_prompt = ChatPromptTemplate.from_template(
    "Write a 3-line blog using this title: {title}"
)

# title -> prompt -> Ollama -> blog text
blog_chain = blog_prompt | llm


# -----------------------------------
# 3. Create Sequential Chain
# -----------------------------------

chain = (
    # Keep original input and generate title
    RunnablePassthrough.assign(
        title=lambda x: title_chain.invoke(
            {"topic": x["topic"]}
        ).content
    )

    # Use generated title to generate blog
    | RunnablePassthrough.assign(
        blog=lambda x: blog_chain.invoke(
            {"title": x["title"]}
        ).content
    )
)


# -----------------------------------
# 4. Run the complete chain
# -----------------------------------

result = chain.invoke({
    "topic": "LLM"
})


# -----------------------------------
# 5. Print outputs
# -----------------------------------

print("TITLE:")
print(result["title"])

print("\nBLOG:")
print(result["blog"])

TITLE:
"Large Language Models: Transforming Text Processing"

BLOG:
Large Language Models (LLMs) are revolutionizing text processing by offering unprecedented capabilities in understanding and generating human-like text. These models, trained on vast amounts of data, can now handle complex tasks such as translation, summarization, and even creative writing, making them indispensable tools across various industries. As LLMs continue to evolve, their impact on natural language processing is set to grow exponentially.


In [30]:
# Import Pydantic classes
from pydantic import BaseModel, Field


# Define the exact structure/schema we want from the LLM
class Customer(BaseModel):

    # LLM must return a string for name
    name: str = Field(
        description="Customer name"
    )

    # LLM must return an integer for age
    age: int = Field(
        description="Customer age"
    )


# Tell the LLM:
# "Whatever you answer, return it in Customer schema"
structured_llm = llm.with_structured_output(Customer)


# Ask the LLM
result = structured_llm.invoke(
    "I love cricket, which player is the best?"
)


# Print structured result
print(result)

name='Virat Kohli' age=35


## Runable Branch

In [31]:
# -----------------------------------
# Simple LLM call
# -----------------------------------

weather = "rainy"

print(
    llm.invoke(
        "Suggest a drink for today."
    ).content
)

Sure! Given that it's currently summer in many parts of the world, how about trying a refreshing Mango Lassi? It’s a delightful blend of yogurt, mango, and a touch of sweetness, often spiced with cardamom or fennel seeds to give it an exotic flavor. This drink is not only delicious but also cooling on a hot day.

If you prefer something non-alcoholic, you can make it with just milk and mango puree. For those who enjoy a bit more excitement, a spiked version using a small amount of rum or vodka can be quite enjoyable as well.

Would you like some tips on how to prepare this drink at home?


In [32]:
def suggest_drink(weather: str):

    # If weather is rainy OR cold
    if weather in ["rainy", "cold"]:

        question = "Suggest a warm chai for rainy weather."

    # Otherwise assume hot weather
    else:

        question = "Suggest a cold drink for hot weather."

    # Send selected prompt to Ollama
    return llm.invoke(question).content


# Test different conditions
print(suggest_drink("rainy"))
print(suggest_drink("hot"))

Certainly! A perfect warm chai for rainy weather is the classic Indian Chai Tea. Here’s a simple recipe you can try at home:

### Ingredients:
- 2 cups of water
- 1-inch piece of ginger, peeled and grated (or to taste)
- 4 green cardamom pods, lightly crushed
- 2 cloves
- 1 cinnamon stick
- 1 black tea bag or 1 teaspoon loose black tea leaves
- 1/2 teaspoon ground fennel seeds (optional, for a unique flavor)
- 1 tablespoon sugar or to taste (adjust according to your preference)
- Milk of your choice (dairy, almond, soy, etc.) - about 1 cup

### Instructions:
1. **Boil the Water:** Start by bringing 2 cups of water to a boil in a small pot.
2. **Add Spices and Ginger:** Add the grated ginger, crushed cardamom pods, cloves, cinnamon stick, and fennel seeds (if using) to the boiling water. Let it simmer for about 5 minutes to allow the flavors to infuse.
3. **Add Tea and Sugar:** Remove from heat and add the black tea bag or loose tea leaves along with sugar. Steep for a few more minutes 

In [33]:
suggest_drink("rainy")

"Certainly! A perfect warm chai for rainy weather is the classic Indian Chai Tea. Here’s a simple recipe you can try at home:\n\n### Ingredients:\n- 2 cups of water\n- 1-inch piece of ginger, peeled and grated (or to taste)\n- 4 green cardamom pods, lightly crushed\n- 2 cloves\n- 1 cinnamon stick\n- 1 black tea bag or 1 teaspoon loose black tea leaves\n- 1/2 teaspoon ground fennel seeds (optional, for a unique flavor)\n- 1 tablespoon sugar or to taste (adjust according to your preference)\n- Milk of your choice (dairy, almond, soy, etc.) - about 1 cup\n\n### Instructions:\n1. **Boil the Water:** Start by bringing 2 cups of water to a boil in a small pot.\n2. **Add Spices and Ginger:** Add the grated ginger, crushed cardamom pods, cloves, cinnamon stick, and fennel seeds (if using) to the boiling water. Let it simmer for about 5 minutes to allow the flavors to infuse.\n3. **Add Tea and Sugar:** Remove from heat and add the black tea bag or loose tea leaves along with sugar. Steep for a 

In [34]:
suggest_drink("hot")

"For hot weather, a refreshing and hydrating cold drink is ideal to help you stay cool and hydrated. Here's a simple yet delicious suggestion:\n\n**Watermelon Juice:**\n- **Ingredients:** Fresh watermelon (about 1/2 large melon), a bit of lime juice for zest, and a touch of honey or sugar if desired.\n- **Preparation:** Cut the watermelon into chunks, remove the seeds, and blend until smooth. Strain to remove any pulp if you prefer a smoother texture. Add a squeeze of fresh lime juice and a dash of sweetener (optional) to taste.\n\nWatermelon is not only hydrating due to its high water content but also rich in vitamins A and C, making it a refreshing and nutritious choice for hot weather. Enjoy your drink while staying cool!"

In [35]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch


# -----------------------------------
# 1. Create two different prompts
# -----------------------------------

# This prompt will be used for rainy/cold/winter weather
chai_prompt = ChatPromptTemplate.from_template(
    "Suggest a warm beverage for {weather} weather"
)

# This prompt will be used for other weather conditions
cold_drink_prompt = ChatPromptTemplate.from_template(
    "Suggest a refreshing cold drink for {weather} weather"
)


# -----------------------------------
# 2. Create two chains
# -----------------------------------

# Warm drink chain
chai_chain = chai_prompt | llm

# Cold drink chain
cold_drink_chain = cold_drink_prompt | llm


# -----------------------------------
# 3. Create conditional branch
# -----------------------------------

chain = RunnableBranch(

    # CONDITION:
    # If weather is rainy, cold, or winter
    # execute chai_chain
    (
        lambda x: x["weather"] in {"rainy", "cold", "winter"},
        chai_chain
    ),

    # DEFAULT:
    # If above condition is False,
    # execute cold_drink_chain
    cold_drink_chain
)


# -----------------------------------
# 4. Test the branch
# -----------------------------------

print(
    chain.invoke(
        {"weather": "rainy"}
    ).content
)

print(
    chain.invoke(
        {"weather": "hot"}
    ).content
)

For rainy weather, a warm and comforting beverage can definitely lift your spirits! Here's a suggestion that is both cozy and delicious:

**Spiced Apple Cider**

Ingredients:
- 4-5 apples (peeled, cored, and chopped)
- 1 cinnamon stick
- 2 cloves
- 1 star anise
- A small piece of ginger (about 1 inch), peeled and sliced
- 2 cups apple juice or cider
- 1 cup water
- 1/4 cup brown sugar (or to taste, depending on the sweetness of your apples)
- A dash of nutmeg

Instructions:
1. In a large pot, combine the chopped apples, cinnamon stick, cloves, star anise, ginger, apple juice/cider, and water.
2. Bring the mixture to a boil over medium heat, then reduce the heat to low and let it simmer for about 30 minutes or until the apples are soft.
3. Remove from heat and strain out the solids (you can save the apple pieces for baking if you like).
4. Return the liquid to the pot and stir in the brown sugar until dissolved. Add a dash of nutmeg for extra warmth.
5. Serve hot, garnished with a cinna

## Runnable Parallel

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel


# -----------------------------------
# 1. Create LinkedIn Prompt
# -----------------------------------

linkedin_prompt = ChatPromptTemplate.from_template(
    "Create a LinkedIn post about {topic}"
)


# -----------------------------------
# 2. Create Instagram Prompt
# -----------------------------------

instagram_prompt = ChatPromptTemplate.from_template(
    "Create a 1-line Instagram caption about {topic}"
)


# -----------------------------------
# 3. Create Parallel Workflow
# -----------------------------------

workflow = RunnableParallel(

    # First independent chain
    linkedin=linkedin_prompt | llm,

    # Second independent chain
    instagram=instagram_prompt | llm
)


# -----------------------------------
# 4. Run workflow
# -----------------------------------

result = workflow.invoke(
    {"topic": "AI"}
)


# -----------------------------------
# 5. Print individual outputs
# -----------------------------------

print("LINKEDIN:")
print(result["linkedin"].content)

print("\nINSTAGRAM:")
print(result["instagram"].content)

LINKEDIN:
🚀 Embracing the Future: How Artificial Intelligence is Transforming Industries 🚀

In today's rapidly evolving world, one technology stands at the forefront of innovation and disruption—Artificial Intelligence (AI). From healthcare to finance, from education to manufacturing, AI is reshaping how we live and work.

💡 **Key Trends in AI:**
1. **Automation & Efficiency:** AI-driven automation is streamlining processes across industries, reducing costs, and increasing productivity.
2. **Personalization:** AI enables personalized experiences for customers, enhancing user satisfaction and engagement.
3. **Data-Driven Decisions:** With advanced analytics, businesses can make informed decisions based on real-time data insights.
4. **Innovation in Healthcare:** AI is revolutionizing diagnostics, treatment plans, and patient care, leading to better health outcomes.

🌟 **The Impact of AI:**
- **Healthcare:** AI-powered tools are improving disease detection, drug discovery, and personaliz

## ReAct Loop

In [37]:
import ast
import operator

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun


# -----------------------------------
# 1. Search Tool
# -----------------------------------

search = DuckDuckGoSearchRun()


@tool
def web_search(query: str) -> str:
    """
    Search the internet for current information.
    """
    return search.invoke(query)


# -----------------------------------
# 2. Calculator Tool
# -----------------------------------

@tool
def calculator(expression: str) -> str:
    """
    Perform basic mathematical calculations.
    Example: 2026 - 1971
    """

    # Safer alternative to directly using eval()
    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
    }

    def calculate(node):
        if isinstance(node, ast.Expression):
            return calculate(node.body)

        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                calculate(node.left),
                calculate(node.right)
            )

        raise ValueError("Unsupported expression")

    tree = ast.parse(expression, mode="eval")

    return str(calculate(tree))


# -----------------------------------
# 3. Create ReAct-style Agent
# -----------------------------------

agent = create_agent(
    model=llm,       # Your existing Ollama LLM
    tools=[
        web_search,
        calculator
    ]
)


# -----------------------------------
# 4. Ask Multi-Step Question
# -----------------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "What is Elon Musk's age in 2026? "
                "Search for his birth year first and then "
                "calculate the age using the calculator."
            )
        }
    ]
})


# -----------------------------------
# 5. Print Final Answer
# -----------------------------------

print(result["messages"][-1].content)

Based on the information found, Elon Musk was born in 1971. Using the calculator, we determined that he will be 55 years old in the year 2026.


In [38]:
!pip install -q langchain-chroma chromadb langchain-ollama

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.114 requires urllib3>=2.2.2, but you have urllib3 2.2.1 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2024.10.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


# ------------------------------------------------
# 1. PRIVATE DOCUMENTS
# ------------------------------------------------

# These represent our private knowledge/data
docs = [
    Document(
        page_content="Raju's tapri opens at 7 AM"
    ),
    Document(
        page_content="Raju's special drink is masala chai"
    ),
    Document(
        page_content="Raju also sells bun maska in the evening"
    )
]


# ------------------------------------------------
# 2. EMBEDDING MODEL
# ------------------------------------------------

# Convert text into numerical vectors using Ollama
# This runs locally — no OpenAI API required
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


# ------------------------------------------------
# 3. VECTOR DATABASE
# ------------------------------------------------

vector_store = Chroma(
    collection_name="chai_notes",
    embedding_function=embeddings
)


# Store our documents in ChromaDB
vector_store.add_documents(docs)


# ------------------------------------------------
# 4. CREATE RETRIEVER
# ------------------------------------------------

# k=1 means retrieve the 1 most relevant document
retriever = vector_store.as_retriever(
    search_kwargs={"k": 1}
)


# ------------------------------------------------
# 5. RETRIEVE RELEVANT DOCUMENT
# ------------------------------------------------

question = "What is Raju's special drink?"

matches = retriever.invoke(question)


# ------------------------------------------------
# 6. CREATE CONTEXT
# ------------------------------------------------

# Extract text from retrieved Documents
context = "\n".join(
    doc.page_content for doc in matches
)


# ------------------------------------------------
# 7. AUGMENT PROMPT WITH RETRIEVED CONTEXT
# ------------------------------------------------

response = llm.invoke(
    f"""
    Use only the following context to answer the question.

    Context:
    {context}

    Question:
    {question}
    """
)


# ------------------------------------------------
# 8. FINAL ANSWER
# ------------------------------------------------

print(response.content)

Raju's special drink is masala chai.
